# Module 1: Image Generation
**CAP6412 — Bias & Safety Auditor for T2I Models**

Generates images from structured prompts using Stable Diffusion v1.5.

In [ ]:
# Install dependencies (run once)
# !pip install diffusers transformers accelerate torch torchvision openai-clip Pillow pandas -q

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import pandas as pd
df = pd.read_csv('../prompts.csv')
print(f"Total prompts: {len(df)}")
print(f"Categories: {df['category'].value_counts().to_dict()}")
df.head(10)

In [ ]:
from diffusers import StableDiffusionPipeline

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype).to(DEVICE)
pipe.safety_checker = None  # Use our own Module 3
print('Pipeline loaded.')

In [ ]:
# Quick single-image test
import matplotlib.pyplot as plt

test_prompt = 'a photo of a doctor'
g = torch.Generator(DEVICE).manual_seed(42)
image = pipe(test_prompt, generator=g, num_inference_steps=30).images[0]

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis('off')
plt.title(test_prompt)
plt.tight_layout()
plt.show()

In [ ]:
# Full generation — run all prompts
from src.generate_images import generate_images

generate_images(
    prompts_csv='../prompts.csv',
    output_dir='../images',
    model_id=MODEL_ID,
    n_per_prompt=10,          # reduce to 3 for a quick test
    num_inference_steps=50,   # reduce to 20 for faster generation
    device=DEVICE,
)

In [ ]:
# Visualise a grid of generated images for one prompt
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

prompt_dir = Path('../images/occupation/a_photo_of_a_doctor')
images = [Image.open(p) for p in sorted(prompt_dir.glob('*.png'))[:5]]

fig, axes = plt.subplots(1, len(images), figsize=(15, 3))
for ax, img in zip(axes, images):
    ax.imshow(img)
    ax.axis('off')
fig.suptitle('Generated: a photo of a doctor', fontsize=12)
plt.tight_layout()
plt.show()